# ETH/USDT historical order book

Inspect **OKX ETH-USDT spot, 20 September 2026 (UTC)**. On 22 September, OKX's
archive index had no September 21 file; September 20 was its newest listed day.
The complete daily file is saved locally. This notebook examines its first **10,000
messages**, about six minutes, using the existing OKX book reader.

Questions: How wide is the spread? How much ETH is visible near the best prices?
Does one side have more size than the other? These are descriptive charts, not trades.

**Run All** reads saved files only. It does not download another day or substitute
sample prices. The archive has exchange timestamps, but no original receipt times
or sequence numbers. The current strategy backtester does not accept L2 books,
so this study stops at data inspection and reports no strategy return.

## 1. Choose the saved data

The paths below select the exact download. After downloading a different day, change
both paths together. Download commands are in the [notebook guide](notebook-operations.md#historical-order-book-study).

In [ ]:
# standard library imports
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
CONFIG = root / "configs/history/okx-eth-usdt.json"
ARCHIVE = root / "data/okx/history/f62e64b7-1b40-4869-8b05-2ca911fefcef"

In [ ]:
# standard library imports
import hashlib
import json
import os
import platform
import tarfile
import time
from datetime import UTC, datetime
from fractions import Fraction
from uuid import uuid4

# third party library imports
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# project library imports
import market_data
from configuration import OKXHistoryConfig
from market_data.adapters.venues.okx import OKXArchiveStore, OKXBook
from market_data.contracts import OKXArchiveRequest, RawObject
from market_data.storage import RawStore
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

if platform.python_version() != (root / ".python-version").read_text().strip():
    raise RuntimeError("Select the repository's pinned Python kernel.")
if "site-packages" not in Path(market_data.__file__).resolve().parts:
    raise RuntimeError("Install the repository package before running this notebook.")
installed = dict(RuntimeFiles.inputs())
source = {
    p.relative_to(root / "src").as_posix(): p
    for package in RuntimeFiles.roots
    for p in (root / "src" / package).rglob("*")
    if p.is_file() and (p.suffix in {".py", ".json"} or p.name == "py.typed")
}
if source.keys() != installed.keys() or any(
    p.read_bytes() != installed[name].read_bytes() for name, p in source.items()
):
    raise RuntimeError("The installed package differs from the checkout; reinstall and restart.")
codec = DomainCodec()
config = OKXHistoryConfig.read(CONFIG)
request = codec.decode_json((ARCHIVE / "request.json").read_bytes(), OKXArchiveRequest)
if request.config != config:
    raise ValueError("The saved archive does not match CONFIG. Select matching paths.")
attempt = os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", str(uuid4()))
output = root / "examples/runs/orderbook-research" / f"{attempt}-artifacts"
output.mkdir(parents=True, exist_ok=False)
(output / "config.json").write_bytes(codec.encode_json(config))
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/orderbook-research.ipynb")
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(c["cell_type"], "".join(c["source"])) for c in notebook["cells"]]
(output / "attempt.json").write_bytes(
    codec.encode_json(
        {
            "question": "Describe spread and the size at the best ten bid and ask levels.",
            "selection": str(ARCHIVE),
            "selected_at": datetime.now(UTC).isoformat(),
            "python": platform.python_version(),
            "implementation": ImplementationIdentity.sha256(),
            "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
            "notebook_source_sha256": hashlib.sha256(
                codec.encode_json(notebook_source)
            ).hexdigest(),
            "strategy_run": False,
        }
    )
)
print(config.symbol, config.day, "UTC")
print("Output:", output)

## 2. Verify and reopen the archive

The installed reader checks the saved file's hash, compression and archive structure,
then rebuilds the first 10,000 book messages. It does not fill gaps or invent timestamps.
A failure stops this notebook.

In [ ]:
summary = OKXArchiveStore(ARCHIVE).replay()
(output / "replay.json").write_bytes(codec.encode_json(summary))
descriptor = codec.decode_json((ARCHIVE / "manifest.json").read_bytes(), RawObject)
epoch = pd.Timestamp("1970-01-01", tz="UTC")
display(
    pd.DataFrame(
        [
            {
                "pair": summary["symbol"],
                "day_UTC": str(config.day),
                "download_MiB": round(summary["saved_bytes"] / 1024**2, 2),
                "levels_per_side": summary["depth"],
                "messages_examined": summary["records"],
                "sample_start_UTC": pd.to_datetime(
                    int(summary["first_source_ms"]), unit="ms", utc=True
                ),
                "sample_end_UTC": pd.to_datetime(
                    int(summary["last_source_ms"]), unit="ms", utc=True
                ),
                "whole_day_examined": summary["entire_archive_replayed"],
                "receipt_times_available": summary["record_receipt_time"] is not None,
            }
        ]
    )
)

## 3. Measure the visible book

Apply the same original messages through `OKXBook`. At each message, record the best
bid and ask, their midpoint, the spread, and the total ETH at the ten best levels on
each side. Size imbalance is `(bid size − ask size) / (bid size + ask size)`.

Prices and sizes use exact fractions for these calculations. Floating-point conversion
is only for the displayed statistics and charts. Empty sides remain missing; crossed
books are counted and shown, not silently removed. Each message has equal weight in
the descriptive statistics; these are not time-weighted averages.

In [ ]:
book = OKXBook(config.symbol)
trace = hashlib.sha256()
records = []
deadline = time.monotonic() + config.duration_seconds
with RawStore(ARCHIVE).open(descriptor=descriptor, request=request.binding) as saved:
    with tarfile.open(fileobj=saved, mode="r|gz") as archive:
        member = archive.next()
        if member is None or member.name != config.filename.removesuffix(".tar.gz") + ".data":
            raise ValueError("Unexpected archive member.")
        stream = archive.extractfile(member)
        if stream is None:
            raise ValueError("The archive has no data stream.")
        with stream:
            for position in range(summary["records"]):
                if time.monotonic() > deadline:
                    raise TimeoutError("Book inspection exceeded the configured time limit.")
                raw = stream.readline(config.max_line_bytes + 1)
                if not raw or len(raw) > config.max_line_bytes or not raw.endswith(b"\n"):
                    raise ValueError("Missing, oversized or incomplete book message.")
                outcome = book.apply_archive(raw)
                trace.update(codec.encode_json((position + 1, raw, outcome, book.identity)))
                source_ms = None if book.source_time is None else int(book.source_time)
                if (
                    source_ms is not None
                    and not config.start_ms <= source_ms < config.start_ms + 86_400_000
                ):
                    raise ValueError("Book timestamp lies outside the selected UTC day.")
                bids = sorted(book.bids, reverse=True)[:10]
                asks = sorted(book.asks)[:10]
                bid = Fraction(bids[0]) if bids else None
                ask = Fraction(asks[0]) if asks else None
                bid_size = sum((Fraction(book.bids[p].size) for p in bids), Fraction(0))
                ask_size = sum((Fraction(book.asks[p].size) for p in asks), Fraction(0))
                total = bid_size + ask_size
                complete = bid is not None and ask is not None
                values = {
                    "bid_USDT": bid,
                    "ask_USDT": ask,
                    "mid_USDT": (bid + ask) / 2 if complete else None,
                    "spread_USDT": ask - bid if complete else None,
                    "bid_top10_ETH": bid_size if bids else None,
                    "ask_top10_ETH": ask_size if asks else None,
                    "size_imbalance": (bid_size - ask_size) / total if complete and total else None,
                }
                records.append(
                    {
                        "position": position,
                        "source_ms": source_ms,
                        "action": outcome,
                        "bid_levels": len(book.bids),
                        "ask_levels": len(book.asks),
                        **{
                            name: None if value is None else str(value)
                            for name, value in values.items()
                        },
                    }
                )
if book.identity != summary["book_identity"] or trace.hexdigest() != summary["trace_identity"]:
    raise AssertionError("Chart input differs from the installed reader's replay.")
with (output / "book-observations.jsonl").open("xb") as retained:
    for row in records:
        retained.write(codec.encode_json(row) + b"\n")
frame = pd.DataFrame(records)
frame["time_UTC"] = pd.to_datetime(frame["source_ms"], unit="ms", utc=True)
metric_columns = list(values)
for name in metric_columns:
    frame[name] = frame[name].map(
        lambda value: float(Fraction(value)) if value is not None else float("nan")
    )
print("Reconstructed messages:", len(frame))
print("Crossed-book observations:", int((frame["spread_USDT"] < 0).sum()))
print("Messages with a missing book side:", int(frame["spread_USDT"].isna().sum()))
display(frame[["time_UTC", *metric_columns]].head(5))
display(frame[metric_columns].describe().round(6))

## 4. Plot prices, spread and size imbalance

The x-axis uses the exchange timestamps supplied by the archive. These charts describe
only the examined prefix, not the whole day. No price is forward-filled.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].plot(frame["time_UTC"], frame["bid_USDT"], label="Best bid", linewidth=0.8)
axes[0].plot(frame["time_UTC"], frame["ask_USDT"], label="Best ask", linewidth=0.8)
axes[0].set_ylabel("USDT per ETH")
axes[0].legend(loc="upper left")
axes[0].set_title(f"OKX {config.symbol} · {config.day} · first {len(frame):,} messages")
axes[1].plot(frame["time_UTC"], frame["spread_USDT"], color="#94612f", linewidth=0.8)
axes[1].set_ylabel("Spread (USDT)")
axes[2].plot(frame["time_UTC"], frame["size_imbalance"], color="#2c7a70", linewidth=0.8)
axes[2].axhline(0, color="grey", linewidth=0.5)
axes[2].set_ylabel("Top-10 size imbalance")
axes[2].set_xlabel("Exchange time (UTC)")
axes[2].set_ylim(-1.05, 1.05)
for axis in axes:
    axis.grid(alpha=0.2)
fig.savefig(output / "orderbook.png", dpi=150)
plt.show()

## 5. What can I do next?

Use the saved observations to investigate spread and depth, or change the calculation
above to try another book feature. This is an initial inspection, with no fitted model
or reserved test period.

**Strategy backtest: not run.** The current engine consumes trade prices, not L2 books.
This archive also lacks original receipt timestamps and sequence IDs. Converting book
updates into trades or treating download time as historical availability would give a
misleading backtest. A book-based simulation needs separate implementation.

In [ ]:
status = {
    "archive": str(ARCHIVE),
    "source_url": request.url,
    "source_sha256": descriptor.sha256,
    "pair": config.symbol,
    "day_UTC": str(config.day),
    "full_archive_saved": True,
    "messages_examined": len(records),
    "whole_day_examined": summary["entire_archive_replayed"],
    "replay_matches_installed_reader": True,
    "strategy_backtest": "not run: current engine does not consume L2 books",
    "strategy_PnL": None,
    "original_receipt_times": "unknown",
    "sequence_continuity": summary["continuity"],
    "output": str(output),
}
(output / "result.json").write_bytes(codec.encode_json(status))
print("Saved charts and observations:", output)
print(status["strategy_backtest"])